# Lendo o arquivo bruto

In [0]:
#Lendo o arquivo json que está no datalake
clientes = spark.read.option("multiline", "true").json("/Volumes/lh_nautical/datalake/datas/clientes_crm.json")
#Mostrando todo o conteúdo do arquivo
clientes.display()

# Salvando o arquivo bruto como tabela delta na camada Bronze

In [0]:
%sql
-- Acessando o catalog lh_nautical
USE CATALOG lh_nautical;
-- Acessando o schema bronze
USE SCHEMA bronze_lh_nautical;


In [0]:
# Salvando o arquivo como tabela na camada bronze 
clientes.write.mode("overwrite").format("delta").saveAsTable("bronze_lh_nautical.tbl_clientes")

# Conferindo a tabela delta criada na camada Bronze


In [0]:
#Conferindo o conteúdo da tabela
tbl_clientes = spark.read.table("bronze_lh_nautical.tbl_clientes")
tbl_clientes.display()

# Recriando a tabela para camada Silver

### - Tratando email

In [0]:
# importacão de algumas bibliotecas
from pyspark.sql.functions import col, trim, lower, regexp_replace
from pyspark.sql.functions import when, col

# Lendo a tabela que está na camada bronze
tbl_clientes_clean = spark.read.table("bronze_lh_nautical.tbl_clientes")
# Alterando todos os email que estão com # por @
tbl_clientes_clean_email = tbl_clientes_clean.withColumn("email", regexp_replace(col("email"), "#", "@"))
tbl_clientes_clean_email.display()
#Salvando a tabela na camada silver
tbl_clientes_clean_email.write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_clientes")



In [0]:
#Conferindo o conteúdo da tabela
tbl_clientes = spark.read.table("silver_lh_nautical.tbl_clientes")
tbl_clientes.display()

### - Separando e tratando a tabela location

In [0]:
# importacão de algumas bibliotecas
from pyspark.sql.functions import col, trim, lower, regexp_replace, length, substring_index
from pyspark.sql.functions import when, col
from pyspark.sql import functions as F
from pyspark.sql.types import *
#Padronizando a coluna location
tbl_clientes_clean_location = spark.read.table("silver_lh_nautical.tbl_clientes")
tbl_clientes_clean_location = tbl_clientes_clean_location.withColumn("location", regexp_replace(col("location"), ",", "-")).withColumn("location", regexp_replace(col("location"), "/", "-"))
tbl_clientes_clean_location.display()
tbl_clientes_clean_location.write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_clientes")

#Separando o estado da cidade
tbl_clientes_clean_location = tbl_clientes_clean_location.withColumn("states", F.when(F.length(F.substring_index(tbl_clientes_clean_location["location"], "-", 1)) > 3, F.substring_index(tbl_clientes_clean_location["location"], "-", -1)).otherwise(F.substring_index(tbl_clientes_clean_location["location"], "-", 1)))

tbl_clientes_clean_location = tbl_clientes_clean_location.withColumn("cities", F.when(F.length(F.substring_index(tbl_clientes_clean_location["location"], "-", 1)) > 3, F.substring_index(tbl_clientes_clean_location["location"], "-", 1)).otherwise(F.substring_index(tbl_clientes_clean_location["location"], "-", -1)))
#Removendo a coluna location
tbl_clientes_clean_location = tbl_clientes_clean_location.drop("location")

#Salvando a modificação
tbl_clientes_clean_location.write.mode("overwrite").option("OverwriteSchema", "true").format("delta").saveAsTable("lh_nautical.silver_lh_nautical.tbl_clientes")
new_tbl_clientes = spark.read.table("lh_nautical.silver_lh_nautical.tbl_clientes")

new_tbl_clientes.display()








In [0]:
#Conferindo o conteúdo da tabela
new_tbl_clientes = spark.read.table("silver_lh_nautical.tbl_clientes")
new_tbl_clientes.display()


### - Reitrando espaços no inicio

In [0]:
new_tbl_clientes.display()
#Removendo os espaços iniciais e finais dos dados das colunas states e cities
new_tbl_clientes = new_tbl_clientes.withColumn("states", trim(new_tbl_clientes["states"])).withColumn("cities", trim(new_tbl_clientes["cities"]))
#Salvando a modificação
new_tbl_clientes.write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_clientes")

